# Build a RAG Agent Lab

<div class="alert alert-block alert-info">
    <b>Note:</b> 
    <p>
        This lab is a direct copy of the
        <a href="https://mastra.ai/en/guides/guide/research-assistant">Mastra Research Assistant Guide</a> 
        I have only expanded upon these concepts with my own thoughts and references.
    </p>
    <p>
        Before starting this lab you will need an OpenAi API Key and a Postgres Connection String.
    </p>
</div>

This lab will show you how to use Mastra to create a Retrieval-Augmented Generation (RAG) system. These systems 
incorporate resource retrieval mechanisms that allow the Ai to gain more context by querying external sources like
the internet or internal tooling. In it you will learn how to provide a text based document to a Vector Store and
query it.

In Jupyter notebooks (including those running JavaScript or TypeScript via tslab), each code cell is often executed in its own scope. This means that variables defined in one cell may not be accessible in another, which can be frustrating when building up a project interactively.
 
To address this, you can use the `globalThis` object. `globalThis` is a standard JavaScript object that provides a universal way to access the global scope, regardless of the environment (browser, Node.js, etc.). By attaching variables to `globalThis`, you ensure they persist and are accessible across all cells in your notebook.
 
**Example:**
 
```typescript
// In one cell
globalThis.pgVector = require('@mastra/pg').pgVector;

// In another cell
const { pgVector } = globalThis;
```

This approach helps you avoid "not defined" errors when referencing variables or modules across different cells. It's especially useful for sharing configuration, database connections, or utility functions throughout your notebook.


---

# Table of Contents

# RAG System Components

Building RAG systems for agents with Mastra requires three components:

1. **Knowledge Store/Index** - This creates numerical representations of textual content.
2. **Retriever** - This matches the result of embedding a query with stored vectors. [More on embeddings later](#embeddings)
3. **Generator** - Creates contextually informed responses using a LLM.

# Project Setup

The easiest way to scaffold this project is by using the `npx create-mastra@latest` `bash` command, then you'll need 
to install several additional modules. These have already been installed for this project, so you don't need to run 
this code.

``````

> npx create-mastra@latest
> npm install @mastra/rag@latest @mastra/pg@latest ai@latest

## Add Keys

You need to add an API provider key and Postgres Connection Key for this lab to work.

Presently, this lab is hard coded to use `openai` as the LLM provider.

Add `OPEN_API_KEY` AND `POSTGRES_CONNECTION_STRING` to the `.env` file.

In [ ]:
// const OPEN_API_KEY = Deno.env.get("OPENAI_API_KEY");
// const POSTGRES_CONNECTION_STRING = Deno.env.get("POSTGRES_CONNECTION_STRING");

const OPENAI_API_KEY = process.env.OPENAI_API_KEY;
const POSTGRES_CONNECTION_STRING = process.env.POSTGRES_CONNECTION_STRING;


# Create the Agent

The Mastra agent created in this lab will use a Vector Query Tool to perform semantic searches over a given vector store to find relevant content. To perform these actios we will need to give the agent 

1. A vector querying tool
2. A LLM to understand queries and generate responses
3. Custom instructions to guide the agent on how to analyze papers, use retrieved content, and acknowledge limitations


### Agent Code

We can begin by creating the query tool and the research agent.

In [ ]:
import { createVectorQueryTool } from '@mastra/rag';
import { Agent } from '@mastra/core/agent';
import { openai } from '@ai-sdk/openai';


const vectorQueryTool = createVectorQueryTool({
    vectorStoreName: "pgVector",
    indexName: "papers",
    model: openai.embedding("text-embedding-3-small"),
});

const researchAgent = new Agent({
    name: "Research Agent",
    instructions: "Vector stores are specialized databases designed to handle and store high-dimensional vector data, which are essentially arrays of numbers representing complex data types like text, images, or audio",
    model: openai("gpt-4.1-mini"),
    tools: { vectorQueryTool }
})
globalThis.researchAgent = researchAgent;

# Create Vector Store

Vector stores are specialized databases designed to handle and store high-dimensional data, that is essentially arrays of numbers that representing complex data types like text, images, or audio. In this lab we use Postgres and [PGVector](https://github.com/pgvector/pgvector).

In [ ]:
import { PgVector } from '@mastra/pg'

const pgVector = new PgVector({
    // connectionString: Deno.env.get("POSTGRES_CONNECTION_STRING")
    connectionString: process.env.POSTGRES_CONNECTION_STRING
});
globalThis.pgVector = pgVector;

In [ ]:
import { Client } from 'pg';

const client = new Client({
  host: 'localhost',
  port: 5432,
  database: 'rag_research_db',
  user: 'rag_user',
  password: 'rag_password'
});

try {
  await client.connect();
  const res = await client.query('SELECT NOW()');
  console.log('Connected successfully:', res.rows[0]);
  await client.end();
} catch (err) {
  console.error('Connection error:', err);
}

# Create Mastra Instance

This lab uses a Mastra agent and gives it access to a Vector store in four lines.

In [ ]:
import { Mastra } from '@mastra/core';


const mastra = new Mastra({
    agents: { researchAgent },
    vectors: { pgVector },
});


# Parse Research Article

The goal of this lab is to create an Ai agent that returns information for a given research article. We need to transform our content into a Mastra readable text format. There are several to choose from, but in this case we can use the `.fromText()` method.

In [ ]:
import { MDocument } from '@mastra/rag';
// This is a workaround to allow SSL connections to be made to the arXiv website
// It's opens a huge vulnerability. 
// Don't do this in production.
process.env.NODE_TLS_REJECT_UNAUTHORIZED = '0';

const paperURL = "https://arxiv.org/html/1706.03762";
const response = await fetch(paperURL);
const paperText = await response.text();

const doc = MDocument.fromText(paperText)

Now that we have data ready to be ingested, it is important to break the document into smaller segments. This is called 
Chunking, and is a best practice in Ai powered workflow design. Chunking promotes efficient retrieval, improved 
embedding quality, flexibility in processing, and metadata extraction.

In [ ]:
const chunks = await doc.chunk({
    strategy: "recursive",
    size: 512,
    overlap: 50,
    separator: "\n",
})
chunks.length

| Parameter | Type | Default | Description |
| --- | --- | --- | --- |
| strategy? | 'recursive' | 'character' | 'token' | 'markdown' | 'html' | 'json' | 'latex' | Defaults based on document type (e.g., .md → 'markdown', .html/.htm → 'html', .json → 'json', .tex → 'latex', others → 'recursive') | The chunking strategy to use. Depending on the strategy, additional optionals may be applied. |
| size? | number | 512 | Maximum size of each chunk. |
| overlap? | number | 50 | Number of characters/tokens that overlap between chunks. |
| separator? | string | \n\n | Character(s) to split on. Defaults to a double newline for text content. |

After chunking the document we need to create embeddings. Embeddings are a core concept to RAG systems. THey are numerical vector representations of text that capture the semantic meaning of content. 

In [ ]:
import { embedMany } from 'ai'

const { embeddings } = await embedMany({
    model: openai.embedding("text-embedding-3-small"),
    values: chunks.map((chunk) => chunk.text)
});

Next, create an index in the Vector Store  

In [ ]:
const vectorStore = mastra.getVector('pgVector')

await vectorStore.createIndex({
    indexName: "papers",
    dimension: 1536,
})


Use the `upsert` method to cerate a new record in a database or update an existing record.

In [ ]:
await vectorStore.upsert({
    indexName: "papers",
    vectors: embeddings,
    metadata: chunks.map((chunk) => ({
        text: chunk.text,
        source: "transformer-paper"
    })),
})

If you've made it this far, then you can finally test the assistant

In [ ]:

const agent = mastra.getAgent("researchAgent");

// Basic query about concepts
const query1 =
    "What problems does sequence modeling face with neural networks?";
const response1 = await agent.generate(query1);
console.log("\nQuery:", query1);
console.log("Response:", response1.text);

# Conclusion

This walkthrough 